# House Price Prediction – Experiment Notebook

Use this notebook to **experiment with the src modules** without touching the production pipeline.

## How outputs are separated from the pipeline

| | Pipeline (`src/main.py`) | This notebook |
|---|---|---|
| Processed data | `data/processed/` | `notebook/outputs/processed/` |
| Model artifact | `models/model.joblib` | `notebook/outputs/models/` |
| Predictions | `reports/predictions.csv` | `notebook/outputs/predictions/` |

**Rule:** Never change paths under `data/processed/`, `models/`, or `reports/` in this notebook — those belong to the CI/CD pipeline.

## Workflow
1. Set your experiment config in **Cell 2**
2. Run cells top-to-bottom
3. Compare metrics at the end — tweak and re-run freely

In [ ]:
# ── Cell 1: Environment setup ────────────────────────────────────────────────
# Add the project root to sys.path so `from src.xxx import ...` works from the
# notebook/ directory.

import sys
import os
from pathlib import Path

# Resolve project root (one level up from notebook/)
PROJECT_ROOT = Path(os.path.abspath("")).parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

# Standard libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split

# src modules
from src.load_data import load_dataset
from src.clean_data import clean_housing_data
from src.validate import validate_dataframe
from src.features import get_feature_preprocessor
from src.train import train_model
from src.evaluate import evaluate_regression
from src.infer import run_inference
from src.utils import ensure_parent_dir

print("All imports OK")

In [ ]:
# ── Cell 2: Experiment config ─────────────────────────────────────────────────
# >>>  THIS IS THE ONLY CELL YOU NEED TO EDIT FOR A NEW EXPERIMENT  <<<
#
# Output paths are intentionally under notebook/outputs/ so they NEVER
# overwrite pipeline artifacts in data/processed/, models/, or reports/.

EXP_SETTINGS = {
    # ── Problem type ─────────────────────────────────────────────────────────
    "problem_type": "regression",       # "regression" | "classification"
    "target_column": "SalePrice",
    "id_column": "Id",

    # ── Input data (reads from the shared raw data; do NOT change these) ─────
    "paths": {
        "train_csv": str(PROJECT_ROOT / "data" / "raw" / "train.csv"),
        "test_csv":  str(PROJECT_ROOT / "data" / "raw" / "test.csv"),   # optional

        # !! OUTPUT PATHS — isolated from the production pipeline !!
        "exp_processed_csv":  str(PROJECT_ROOT / "notebook" / "outputs" / "processed" / "clean.csv"),
        "exp_model_artifact": str(PROJECT_ROOT / "notebook" / "outputs" / "models"    / "model.joblib"),
        "exp_predictions_csv":str(PROJECT_ROOT / "notebook" / "outputs" / "predictions" / "predictions.csv"),
    },

    # ── Train / test split ────────────────────────────────────────────────────
    "split": {
        "test_size":    0.2,
        "random_state": 42,
    },

    # ── Feature config ────────────────────────────────────────────────────────
    # Edit these lists to try different feature groupings.
    # quantile_bin     → numeric cols binned into n_bins quantile buckets
    # categorical_ohe  → categorical cols one-hot encoded
    # numeric_scaled   → numeric cols median-imputed + StandardScaled
    "features": {
        "quantile_bin":       ["LotArea", "GrLivArea"],
        "categorical_onehot": ["Neighborhood"],
        "numeric_passthrough": ["OverallQual", "YearBuilt"],
        "n_bins": 4,               # experiment: try 3, 4, 5, 10 …
    },

    # ── Cleaning ──────────────────────────────────────────────────────────────
    "cleaning": {
        "drop_cols": ["Id"],
    },
}

# Create all output directories up front
for key in ["exp_processed_csv", "exp_model_artifact", "exp_predictions_csv"]:
    Path(EXP_SETTINGS["paths"][key]).parent.mkdir(parents=True, exist_ok=True)

print("Experiment outputs will be written to:")
for k, v in EXP_SETTINGS["paths"].items():
    if k.startswith("exp_"):
        print(f"  {k}: {v}")

---
## Step 1 — Load data
Calls `src.load_data.load_dataset`. Returns a dict with `"train"` (and optionally `"test"`) `LoadResult` objects.

In [ ]:
# ── Cell 3: Load raw data ─────────────────────────────────────────────────────
train_path = Path(EXP_SETTINGS["paths"]["train_csv"])
test_path  = Path(EXP_SETTINGS["paths"]["test_csv"])

data = load_dataset(train_path, test_path if test_path.exists() else None)

df_train_raw = data["train"].df
df_test_raw  = data["test"].df if "test" in data else None

print(f"Train shape : {df_train_raw.shape}")
if df_test_raw is not None:
    print(f"Test shape  : {df_test_raw.shape}")
df_train_raw.head(3)

---
## Step 2 — Exploratory Data Analysis
Quick sanity checks before cleaning. Adjust the plots below as needed.

In [ ]:
# ── Cell 4: Target distribution ───────────────────────────────────────────────
target = EXP_SETTINGS["target_column"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df_train_raw[target], bins=50, edgecolor="white", color="steelblue")
axes[0].set_title(f"{target} – raw")
axes[0].set_xlabel(target)

axes[1].hist(np.log1p(df_train_raw[target]), bins=50, edgecolor="white", color="seagreen")
axes[1].set_title(f"log1p({target})")
axes[1].set_xlabel(f"log1p({target})")

plt.tight_layout()
plt.show()

print(f"Raw skew   : {df_train_raw[target].skew():.3f}")
print(f"Log1p skew : {np.log1p(df_train_raw[target]).skew():.3f}")

In [ ]:
# ── Cell 5: Missing values overview ──────────────────────────────────────────
missing = df_train_raw.isnull().mean().sort_values(ascending=False)
missing_top = missing[missing > 0].head(20)

if missing_top.empty:
    print("No missing values in the training set.")
else:
    missing_top.plot(kind="bar", figsize=(12, 4), color="salmon", edgecolor="white")
    plt.title("Top columns with missing values (% of rows)")
    plt.ylabel("Missing fraction")
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Cell 6: Correlation heatmap (numeric features) ───────────────────────────
numeric_cols = df_train_raw.select_dtypes(include=[np.number]).columns.tolist()
corr_with_target = (
    df_train_raw[numeric_cols]
    .corr()[[target]]
    .drop(target)
    .sort_values(target, ascending=False)
)

top_n = 15
top_corr = pd.concat([corr_with_target.head(top_n), corr_with_target.tail(top_n)])

plt.figure(figsize=(6, 8))
sns.heatmap(
    top_corr,
    annot=True, fmt=".2f", cmap="RdYlGn",
    center=0, vmin=-1, vmax=1, linewidths=0.5,
)
plt.title(f"Top / bottom {top_n} correlations with {target}")
plt.tight_layout()
plt.show()

---
## Step 3 — Clean data
`src.clean_data.clean_housing_data` standardises column names, removes duplicates, drops specified columns, and separates X from y.

In [ ]:
# ── Cell 7: Clean ─────────────────────────────────────────────────────────────
clean_result = clean_housing_data(
    df_train_raw,
    target_col=EXP_SETTINGS["target_column"],
    drop_cols=EXP_SETTINGS["cleaning"]["drop_cols"],
    require_target=True,
)

X_all = clean_result.X
y_all = clean_result.y

print(f"X shape : {X_all.shape}")
print(f"y shape : {y_all.shape}")
print(f"Dropped : {clean_result.dropped_columns}")

# Save experiment processed data (isolated path)
exp_clean_path = Path(EXP_SETTINGS["paths"]["exp_processed_csv"])
df_clean = X_all.copy()
df_clean[EXP_SETTINGS["target_column"]] = y_all.values
df_clean.to_csv(exp_clean_path, index=False)
print(f"\nSaved clean data → {exp_clean_path}")

---
## Step 4 — Validate
`src.validate.validate_dataframe` runs domain-constraint checks. Fails fast if the data doesn't meet expectations.

In [ ]:
# ── Cell 8: Validate ──────────────────────────────────────────────────────────
required_cols = list(
    dict.fromkeys(
        EXP_SETTINGS["features"]["quantile_bin"]
        + EXP_SETTINGS["features"]["categorical_onehot"]
        + EXP_SETTINGS["features"]["numeric_passthrough"]
    )
)

# Only validate columns that actually exist in the cleaned df
# (some columns from the full Ames dataset may not be in the dummy/subset)
available_required = [c for c in required_cols if c in X_all.columns]
validate_dataframe(X_all, available_required)
print("Validation passed for required columns:", available_required)

---
## Step 5 — Train / test split

In [ ]:
# ── Cell 9: Split ─────────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all,
    test_size=EXP_SETTINGS["split"]["test_size"],
    random_state=EXP_SETTINGS["split"]["random_state"],
)

print(f"X_train : {X_train.shape}  |  X_test : {X_test.shape}")
print(f"y_train : {y_train.shape}  |  y_test : {y_test.shape}")

---
## Step 6 — Build feature preprocessor
`src.features.get_feature_preprocessor` returns an **unfitted** `ColumnTransformer`.
Fitting happens inside the Pipeline in `train_model` — no leakage.

**Experiment tip:** Edit the feature lists in Cell 2 and re-run from here.

In [ ]:
# ── Cell 10: Preprocessor ────────────────────────────────────────────────────
preprocessor = get_feature_preprocessor(
    quantile_bin_cols=EXP_SETTINGS["features"]["quantile_bin"],
    categorical_onehot_cols=EXP_SETTINGS["features"]["categorical_onehot"],
    numeric_passthrough_cols=EXP_SETTINGS["features"]["numeric_passthrough"],
    n_bins=int(EXP_SETTINGS["features"]["n_bins"]),
)

print("Preprocessor steps:")
for name, pipe, cols in preprocessor.transformers:
    print(f"  {name:25s} → {cols}")

---
## Step 7 — Train model
`src.train.train_model` wraps the preprocessor + Lasso estimator in a `Pipeline` and runs `GridSearchCV`.
Target is modelled in **log-space** (log1p) — `evaluate_regression` and `run_inference` invert this automatically.

In [ ]:
# ── Cell 11: Train ────────────────────────────────────────────────────────────
model = train_model(
    X_train, y_train,
    preprocessor,
    problem_type=EXP_SETTINGS["problem_type"],
)

print(f"Best params : {model.best_params_}")
print(f"Best CV score (neg-RMSE log-space): {model.best_score_:.4f}")

# Save experiment model artifact (isolated path — NOT models/model.joblib)
exp_model_path = Path(EXP_SETTINGS["paths"]["exp_model_artifact"])
artifact = {
    "pipeline": model,
    "metadata": {
        "problem_type": EXP_SETTINGS["problem_type"],
        "target_transform": "log1p" if EXP_SETTINGS["problem_type"] == "regression" else "none",
    },
}
joblib.dump(artifact, exp_model_path)
print(f"\nSaved experiment model → {exp_model_path}")

---
## Step 8 — Evaluate
Predictions are inverted from log-space with `expm1` before computing metrics in the original price scale.

In [ ]:
# ── Cell 12: Evaluate ─────────────────────────────────────────────────────────
y_pred_log   = model.predict(X_test)
y_pred_price = np.expm1(y_pred_log)           # invert log1p
y_true_price = y_test.astype(float).values

metrics = evaluate_regression(y_true_price, y_pred_price, compute_rmsle=True)

print("\n── Experiment metrics (price-scale) ──")
for k, v in metrics.items():
    print(f"  {k:8s}: {v:,.4f}")

In [ ]:
# ── Cell 13: Residual plots ───────────────────────────────────────────────────
residuals = y_true_price - y_pred_price

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 1. Predicted vs actual
axes[0].scatter(y_true_price, y_pred_price, alpha=0.4, s=20, color="steelblue")
lims = [min(y_true_price.min(), y_pred_price.min()),
        max(y_true_price.max(), y_pred_price.max())]
axes[0].plot(lims, lims, "r--", linewidth=1)
axes[0].set_xlabel("Actual")
axes[0].set_ylabel("Predicted")
axes[0].set_title("Predicted vs Actual")

# 2. Residuals vs predicted
axes[1].scatter(y_pred_price, residuals, alpha=0.4, s=20, color="seagreen")
axes[1].axhline(0, color="r", linewidth=1, linestyle="--")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Residual")
axes[1].set_title("Residuals vs Predicted")

# 3. Residual distribution
axes[2].hist(residuals, bins=40, color="salmon", edgecolor="white")
axes[2].axvline(0, color="r", linewidth=1, linestyle="--")
axes[2].set_xlabel("Residual")
axes[2].set_title("Residual Distribution")

plt.suptitle(f"Experiment – R²={metrics['r2']:.3f}  RMSE={metrics['rmse']:,.0f}  MAE={metrics['mae']:,.0f}")
plt.tight_layout()
plt.show()

---
## Step 9 — Cross-validation learning curve (optional)
Inspect how CV score changes with each alpha value from the grid search.

In [ ]:
# ── Cell 14: GridSearch CV results ───────────────────────────────────────────
cv_results = pd.DataFrame(model.cv_results_)
cv_results = cv_results[["param_model__alpha", "mean_test_score", "std_test_score"]].copy()
cv_results["rmse_log"] = -cv_results["mean_test_score"]
cv_results = cv_results.sort_values("param_model__alpha")

plt.figure(figsize=(8, 4))
plt.errorbar(
    cv_results["param_model__alpha"].astype(float),
    cv_results["rmse_log"],
    yerr=cv_results["std_test_score"],
    marker="o", capsize=4, color="steelblue",
)
plt.xscale("log")
plt.xlabel("alpha (log scale)")
plt.ylabel("CV RMSE (log-space target)")
plt.title("Lasso – GridSearch CV scores per alpha")
plt.axvline(model.best_params_["model__alpha"], color="r", linestyle="--", label=f"best alpha={model.best_params_['model__alpha']}")
plt.legend()
plt.tight_layout()
plt.show()

print(cv_results.to_string(index=False))

---
## Step 10 — Feature importance (non-zero Lasso coefficients)

In [ ]:
# ── Cell 15: Feature coefficients ────────────────────────────────────────────
best_pipeline = model.best_estimator_
lasso_step    = best_pipeline.named_steps["model"]
preproc_step  = best_pipeline.named_steps["preprocess"]

# Recover feature names from the fitted ColumnTransformer
try:
    feature_names = preproc_step.get_feature_names_out()
except AttributeError:
    feature_names = [f"feature_{i}" for i in range(len(lasso_step.coef_))]

coef_series = pd.Series(lasso_step.coef_, index=feature_names)
nonzero     = coef_series[coef_series != 0].sort_values(key=abs, ascending=False)

print(f"Non-zero features: {len(nonzero)} / {len(coef_series)}")

top_k = min(30, len(nonzero))
nonzero.head(top_k).sort_values().plot(
    kind="barh", figsize=(8, top_k * 0.3 + 1), color="steelblue", edgecolor="white"
)
plt.xlabel("Lasso coefficient")
plt.title(f"Top {top_k} non-zero Lasso coefficients")
plt.axvline(0, color="k", linewidth=0.8)
plt.tight_layout()
plt.show()

---
## Step 11 — Inference
Run predictions on the held-out test split (or on `data/raw/test.csv` if available).
Output is saved to `notebook/outputs/predictions/` — NOT to `reports/`.

In [ ]:
# ── Cell 16: Inference ────────────────────────────────────────────────────────
if df_test_raw is not None:
    # Use the real test.csv (no target column)
    clean_test = clean_housing_data(
        df_test_raw,
        target_col=EXP_SETTINGS["target_column"],
        drop_cols=EXP_SETTINGS["cleaning"]["drop_cols"],
        require_target=False,
    )
    X_infer = clean_test.X
    print(f"Inferring on test.csv  → {X_infer.shape}")
else:
    # Fall back to held-out test split
    X_infer = X_test.copy()
    print(f"No test.csv found — inferring on X_test split → {X_infer.shape}")

preds_df = run_inference(
    input_df=X_infer,
    artifact=artifact,
    id_col=EXP_SETTINGS["id_column"],
    pred_col=EXP_SETTINGS["target_column"],
)

# Save experiment predictions (isolated path — NOT reports/predictions.csv)
exp_preds_path = Path(EXP_SETTINGS["paths"]["exp_predictions_csv"])
ensure_parent_dir(str(exp_preds_path))
preds_df.to_csv(exp_preds_path, index=False)

print(f"\nSaved predictions → {exp_preds_path}")
preds_df.head(5)

---
## Summary

Re-run the cell below after any experiment to get a quick summary.

In [ ]:
# ── Cell 17: Experiment summary ───────────────────────────────────────────────
print("════════════════════════════════════════")
print("         EXPERIMENT SUMMARY")
print("════════════════════════════════════════")
print(f"Problem type  : {EXP_SETTINGS['problem_type']}")
print(f"Train samples : {X_train.shape[0]}")
print(f"Test  samples : {X_test.shape[0]}")
print(f"Best alpha    : {model.best_params_['model__alpha']}")
print()
print("Metrics (price-scale):")
for k, v in metrics.items():
    print(f"  {k:8s}: {v:,.4f}")
print()
print("Feature config:")
for k, v in EXP_SETTINGS["features"].items():
    print(f"  {k:25s}: {v}")
print()
print("Outputs:")
for k in ["exp_processed_csv", "exp_model_artifact", "exp_predictions_csv"]:
    print(f"  {k}: {EXP_SETTINGS['paths'][k]}")
print("════════════════════════════════════════")